# ????????????

?? `src/fea_cpt` ????????????????????????

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

workspace = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(workspace / 'src'))

from fea_cpt.io_utils import iter_npz_files, load_feature_record
from fea_cpt.params import DEFAULT_FEATURE_PARAMS, build_feature_params
from fea_cpt.pipeline import compute_feature_table_for_paths, validate_feature_table
from fea_cpt.importance import rank_feature_importance
from fea_cpt.signal_ops import build_context


d:\anaconda3\envs\LZdataread39\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ?????
data_root = workspace / 'data'
output_dir = workspace / 'outputs' / 'fea_cpt_notebook'
output_dir.mkdir(parents=True, exist_ok=True)

all_paths = iter_npz_files(data_root)
selected_paths = all_paths[:12]

params = build_feature_params(
    highpass_hz=1_000.0,
    main_band_hz=(5_000.0, 60_000.0),
    harmonic_band_hz=(25_000.0, 60_000.0),
    envelope_smooth_ms=0.2,
    n_jobs=1,
)

save_feature_table = True
save_validation_table = True
save_importance_tables = True


In [3]:
feature_table = compute_feature_table_for_paths(selected_paths, params=params, show_progress=True, log_dir=output_dir / 'logs')
validation_table = validate_feature_table(feature_table)
feature_table.head()


2026-04-19 01:03:47 | INFO | fea_cpt.pipeline | Batch computation started for 12 files
计算特征:   0%|          | 0/12 [00:00<?, ?it/s]2026-04-19 01:03:47 | INFO | fea_cpt.compute | Start feature computation for BK14-FIP-200K-20260323T110106.536.npz
2026-04-19 01:03:50 | INFO | fea_cpt.compute | Finished feature computation for BK14-FIP-200K-20260323T110106.536.npz with 80 features
计算特征:   8%|▊         | 1/12 [00:02<00:30,  2.81s/it]2026-04-19 01:03:50 | INFO | fea_cpt.compute | Start feature computation for BK14-FIP-200K-20260323T120121.475.npz
2026-04-19 01:03:52 | INFO | fea_cpt.compute | Finished feature computation for BK14-FIP-200K-20260323T120121.475.npz with 80 features
计算特征:  17%|█▋        | 2/12 [00:05<00:27,  2.78s/it]2026-04-19 01:03:52 | INFO | fea_cpt.compute | Start feature computation for BK14-FIP-200K-20260323T154514.886.npz
2026-04-19 01:03:55 | INFO | fea_cpt.compute | Finished feature computation for BK14-FIP-200K-20260323T154514.886.npz with 80 features
计算特征:  25%|██▌ 

,sample_id,sample_name,sample_type,sample_type_code,path,r_p,C_E,A_env,S_env,Sk_env,...,R_wp_ddda,R_wp_dddd,H_wp,I_burst,D_WPT,C_damp,alpha_hat,Q_MP,Delta_J,eta_dict
0,BK14-FIP-200K-20260323T110106.536,BK14-FIP-200K-20260323T110106.536.npz,BK14,571,E:\codes\ZZ-BK\data\BK14-0.06S\BK14-FIP-200K-2...,0.180515,0.023768,0.638971,0.047585,2.597016,...,0.000017,0.000020,1.025890,17566.827354,-0.999777,0.002102,-19.745329,0.131419,2.067631e-07,0.000251
1,BK14-FIP-200K-20260323T120121.475,BK14-FIP-200K-20260323T120121.475.npz,BK14,571,E:\codes\ZZ-BK\data\BK14-0.06S\BK14-FIP-200K-2...,0.015329,0.023081,0.969342,0.046490,2.019084,...,0.000008,0.000010,0.989061,21721.787368,-0.999882,0.000619,-18.798908,0.229167,1.842100e-08,0.000048
2,BK14-FIP-200K-20260323T154514.886,BK14-FIP-200K-20260323T154514.886.npz,BK14,571,E:\codes\ZZ-BK\data\BK14-0.06S\BK14-FIP-200K-2...,0.044581,0.024410,0.910838,0.175434,2.130032,...,0.000057,0.000040,1.112737,7439.794485,-0.999524,0.000988,-32.585151,0.154324,9.115415e-08,0.000168
3,BK14-FIP-200K-20260323T175314.547,BK14-FIP-200K-20260323T175314.547.npz,BK14,571,E:\codes\ZZ-BK\data\BK14-0.06S\BK14-FIP-200K-2...,0.183327,0.023350,0.633346,0.105438,2.102004,...,0.000019,0.000036,0.944682,11895.766856,-0.999716,0.000886,-21.507914,0.396566,7.652680e-08,0.000116
4,BK14-FIP-200K-20260323T212024.779,BK14-FIP-200K-20260323T212024.779.npz,BK14,571,E:\codes\ZZ-BK\data\BK14-0.06S\BK14-FIP-200K-2...,0.054817,0.020149,0.890367,-0.061158,2.875414,...,0.000010,0.000003,1.007580,22639.524730,-0.999921,0.001227,2.236872,0.212655,6.126064e-08,0.000137


In [4]:
if save_feature_table:
    feature_table.to_csv(output_dir / 'feature_table.csv', index=False, encoding='utf-8-sig')
    feature_columns = [c for c in feature_table.columns if c not in {'sample_id', 'sample_name', 'sample_type', 'sample_type_code', 'path'}]
    pd.DataFrame({'feature_index': range(len(feature_columns)), 'feature_name': feature_columns}).to_csv(output_dir / 'feature_vector_mapping.csv', index=False, encoding='utf-8-sig')

if save_validation_table:
    validation_table.to_csv(output_dir / 'feature_validation.csv', index=False, encoding='utf-8-sig')

print('feature_table shape =', feature_table.shape)
print('validation_table shape =', validation_table.shape)
validation_table.head(20)


feature_table shape = (12, 85)
validation_table shape = (80, 9)


,feature_name,non_null_count,min,max,mean,std,abs_max_log10,has_nan,has_inf
0,A_env,12,3.563322e-01,1.002943e+00,7.973024e-01,1.970640e-01,1.276295e-03,False,False
1,CF_res,12,7.521890e+00,1.183493e+01,9.880614e+00,1.401619e+00,1.073166e+00,False,False
2,C_E,12,2.014938e-02,2.767158e-02,2.354966e-02,2.215063e-03,-1.557966e+00,False,False
3,C_bulge,12,1.893333e-02,5.160000e-02,3.063611e-02,1.146975e-02,-1.287350e+00,False,False
4,C_damp,12,5.900966e-04,8.682212e-03,2.127028e-03,2.338710e-03,-2.061370e+00,False,False
5,C_f,12,3.195115e+06,5.572739e+06,4.621377e+06,8.635409e+05,6.746069e+00,False,False
6,C_h,12,1.144308e+03,2.496671e+03,2.044084e+03,4.384844e+02,3.397361e+00,False,False
7,D_WPT,12,-9.999999e-01,-9.995244e-01,-9.998921e-01,1.507791e-04,-4.508299e-08,False,False
8,Delta_J,12,1.842100e-08,2.095551e-06,3.370513e-07,6.052621e-07,-5.678702e+00,False,False
9,Delta_f_span,12,3.906250e+03,1.464844e+04,7.373047e+03,2.857327e+03,4.165791e+00,False,False


In [5]:
quick_top = validation_table.sort_values('abs_max_log10', ascending=False).head(20)
print(quick_top[['feature_name', 'mean', 'abs_max_log10']].to_string(index=False))


 feature_name          mean  abs_max_log10
     k_res_hl -2.028008e+11      12.192901
R_res_hl_mean  8.229441e+09      10.820912
         R_td  2.041667e+09      10.389166
          C_f  4.621377e+06       6.746069
      I_burst  1.113705e+06       6.515532
     k_res_sc -1.445520e+05       5.694002
         k_sc -6.246518e+04       5.483706
  SC_res_mean  1.237459e+04       4.278947
 Delta_f_span  7.373047e+03       4.165791
      SC_mean  9.289088e+03       4.144384
       E_harm  2.097576e+03       3.732121
          C_h  2.044084e+03       3.397361
        E_res  2.266760e+02       2.879790
       eta_bw -3.723987e+01       2.857723
       SK_max  3.435201e+02       2.715942
       N_turn  2.915833e+02       2.579784
        N_abn  3.050000e+01       2.173186
       F_peak  4.062530e+01       2.170738
   R_res_tkeo  8.509666e+01       2.050371
       R_tkeo  5.440415e+01       1.895394


In [6]:
meta_cols = ['sample_id', 'sample_name', 'sample_type', 'sample_type_code', 'path']
feature_cols = [c for c in feature_table.columns if c not in meta_cols]

importance_tables = None
if feature_table['sample_type'].nunique() >= 2:
    X = feature_table[feature_cols].fillna(0.0)
    y = feature_table['sample_type_code']
    importance_tables = rank_feature_importance(X, y, task='classification', top_k=20)
    for name, table in importance_tables.items():
        print(f'\n{name} top20')
        print(table.to_string(index=False))
        if save_importance_tables:
            table.to_csv(output_dir / f'importance_{name}.csv', index=False, encoding='utf-8-sig')
else:
    print('???????? 2 ??????????')


???????? 2 ??????????


In [ ]:
record = load_feature_record(selected_paths[0])
context = build_context(record, params)
t = np.arange(len(record.signal)) / record.sample_rate * 1e3

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(t, record.signal, lw=0.8)
axes[0].set_title('Raw Signal')
axes[1].plot(t, context.main_signal, lw=0.8)
axes[1].set_title('Preprocessed Main-Band Signal')
axes[2].plot(t, context.envelope, lw=1.0, label='Envelope')
axes[2].plot(t, context.envelope_fit, lw=1.0, label='Bulge Fit')
axes[2].axvline(context.envelope_peak_index / record.sample_rate * 1e3, color='r', ls='--', lw=0.8)
axes[2].legend()
axes[2].set_title('Envelope and Fitted Bulge')
axes[2].set_xlabel('Time (ms)')
fig.tight_layout()
plt.show()
fig.savefig(output_dir / 'signal_envelope.png', dpi=150, bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
pcm0 = axes[0].pcolormesh(context.stft_times * 1e3, context.stft_freqs / 1e3, 10 * np.log10(context.stft_power + 1e-12), shading='auto')
axes[0].plot(context.stft_times * 1e3, context.ridge_f1 / 1e3, color='w', lw=1.0, label='f1 ridge')
axes[0].plot(context.stft_times * 1e3, context.ridge_f2 / 1e3, color='c', lw=1.0, label='f2 ridge')
axes[0].legend()
axes[0].set_title('Main STFT')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Frequency (kHz)')
fig.colorbar(pcm0, ax=axes[0])

pcm1 = axes[1].pcolormesh(context.stft_times * 1e3, context.stft_freqs / 1e3, 10 * np.log10(context.residual_power + 1e-12), shading='auto')
axes[1].set_title('Residual STFT')
axes[1].set_xlabel('Time (ms)')
fig.colorbar(pcm1, ax=axes[1])
fig.tight_layout()
plt.show()
fig.savefig(output_dir / 'stft_and_residual.png', dpi=150, bbox_inches='tight')
